In [63]:
import pandas as pd

df_train = pd.read_csv('data/train_transaction.csv')

df_train.shape

(590540, 394)

## Ordenar dataset por TransactionDT para luego separar el dataset en splits

In [64]:
df_train = df_train.sort_values('TransactionDT').reset_index(drop=True)

In [65]:
y = df_train['isFraud']
X = df_train.drop(columns=['isFraud'])

print(X.shape)
print(y.shape)

(590540, 393)
(590540,)


In [66]:
X.dtypes.value_counts()

float64    376
str         14
int64        3
Name: count, dtype: int64

In [68]:
X.select_dtypes(include='object').columns

C:\Users\juanc\AppData\Local\Temp\ipykernel_86604\2418744409.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include='object').columns


Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1',
       'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'],
      dtype='str')

M1 a M9 — son columnas de match, sus valores son solo "T" o "F" (verdadero/falso)



In [69]:
X[['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']].value_counts()

M1  M2  M3  M4  M5  M6  M7  M8  M9
T   T   T   M0  T   F   F   F   T     11558
                F   F   F   F   T     10064
                    T   F   F   T      8555
                T   T   F   F   T      7373
                F   F   F   F   F      5734
                                      ...  
F   T   F   M0  F   T   F   F   F         1
T   T   F   M1  F   F   T   T   F         1
        T   M0  T   F   F   T   F         1
    F   F   M0  F   T   T   F   T         1
    T   T   M0  T   T   F   T   F         1
Name: count, Length: 187, dtype: int64

In [70]:
X['M4'].value_counts()

M4
M0    196405
M2     59865
M1     52826
Name: count, dtype: int64

## Tipos de columnas categóricas

### Grupo M — columnas de match
- `M1, M2, M3, M5, M6, M7, M8, M9` → binarias (T/F) → mapeo simple: T=1, F=0
- `M4` → 3 valores (M0, M1, M2) → necesita encoding separado

### Grupo principal — categorías reales
- `ProductCD`, `card4`, `card6` → pocas categorías, valores de negocio
- `P_emaildomain`, `R_emaildomain` → dominios de email, muchos valores únicos posibles

In [71]:
X[['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']] = X[['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9']].apply(lambda x: x.map({'T': 1, 'F': 0}))

In [72]:
X['M4'].value_counts()

M4
M0    196405
M2     59865
M1     52826
Name: count, dtype: int64

## Encoding de M4

`M4` tiene 3 valores: `M0`, `M1`, `M2`. Se aplica **ordinal encoding** (M0→0, M1→1, M2→2) por dos razones:

1. El nombre de los valores sugiere un índice implícito, Verizon los nombró con orden numérico.
2. One-hot encoding generaría columnas adicionales innecesarias para un modelo de árboles como LightGBM, que maneja ordinales bien sin perder información.

> Limitación: no sabemos si el orden real es M0 < M1 < M2. Si el modelo falla, revisar este encoding es un buen punto de partida.

In [73]:
X['M4'] = X['M4'].map(({'M0':0, 'M1':1, 'M2':2}))

In [74]:
X['M4'].value_counts()

M4
0.0    196405
2.0     59865
1.0     52826
Name: count, dtype: int64

## `.apply()` con lambda — comportamiento según el objeto

- `DataFrame.apply(lambda x: ...)` → `x` es una columna completa (Serie); `x.map()` funciona.
- `Serie.apply(lambda x: ...)` → `x` es un valor individual (string, int, etc.); `x.map()` no existe.
- No ovldiar, si voy a manejar columnas, las manejo como columnas/series, y si manejo texto, la manejo como texto antivo.

In [75]:
X[['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']].nunique()

ProductCD         5
card4             4
card6             4
P_emaildomain    59
R_emaildomain    60
dtype: int64

In [76]:
for col in ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']:
    print(f"\n{col}:")
    print(X[col].value_counts())


ProductCD:
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

card4:
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
Name: count, dtype: int64

card6:
card6
debit              439938
credit             148986
debit or credit        30
charge card            15
Name: count, dtype: int64

P_emaildomain:
P_emaildomain
gmail.com           228355
yahoo.com           100934
hotmail.com          45250
anonymous.com        36998
aol.com              28289
comcast.net           7888
icloud.com            6267
outlook.com           5096
msn.com               4092
att.net               4033
live.com              3041
sbcglobal.net         2970
verizon.net           2705
ymail.com             2396
bellsouth.net         1909
yahoo.com.mx          1543
me.com                1522
cox.net               1393
optonline.net         1011
charter.net            816
live.com.mx            749
roc

In [77]:
X.select_dtypes(include='object').columns

C:\Users\juanc\AppData\Local\Temp\ipykernel_86604\2418744409.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include='object').columns


Index(['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain'], dtype='str')

## Split temporal vs. aleatorio

En datos financieros con componente de tiempo, el split debe ser **cronológico**, no aleatorio.

- Split aleatorio → mezcla datos futuros en el entrenamiento → el modelo aprende patrones que en producción no existirían aún → métricas infladas artificialmente (**temporal leakage**).
- Split cronológico → entrena con el pasado, valida con el futuro → simula exactamente el escenario real de producción.

> Regla: siempre predices el futuro con datos del pasado. El split debe reflejar eso.

## Data leakage — regla general

El modelo nunca debe ver información que no tendría disponible en producción al momento de predecir.

Dos formas comunes:
- **Target leakage:** usar la variable objetivo (o derivados de ella) para calcular transformaciones antes de separar train/validation.
- **Temporal leakage:** mezclar datos futuros en el entrenamiento cuando los datos tienen componente de tiempo.

> Solución: hacer el split **antes** de calcular cualquier encoding o transformación que use estadísticas del dataset.

In [ ]:
%pip install scikit-learn

## **Dividir dataset**

Se divide el dataset teniendo ordenado todo por tiempo de la variable TransactionDT, para tener el contexto real de como se decta, no de manera aleatoria

In [79]:
X = X.sort_values('TransactionDT')

corte = int(len(X)*0.8)

X_train = X.iloc[:corte]
y_train = y.iloc[:corte]

X_Val = X.iloc[corte:]
y_Val = y.iloc[corte:]

In [82]:
X_train.shape, X_Val.shape

((472432, 393), (118108, 393))

In [81]:
y_train.shape, y_Val.shape

((472432,), (118108,))